In [144]:
!pip install py7zr
import tensorflow as tf
import os
import zipfile as unzip1
import py7zr as unzip2
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [126]:
with unzip1.ZipFile("14) reviewsdata.zip", mode="r") as archive:
  archive.extractall(".")
with unzip2.SevenZipFile("14) reviewsdata.7z", mode="r") as archive:
  archive.extractall(".")

In [127]:
Exp = [23,43,64,74,-852,65,28]
DS = tf.data.Dataset.from_tensor_slices(Exp)
for i in DS.filter(lambda x: x>0).map(lambda y: y*72).shuffle(2).batch(2).take(2).as_numpy_iterator():
  print(i) # replace i with i.numpy() if as_numpy_iterator() method is not used in iterator.

[1656 4608]
[3096 4680]


In [128]:
def is_non_empty(file_path):
  content = tf.io.read_file(file_path)
  return tf.strings.length(content) > 0
def count(ds):
    return sum(1 for _ in ds)
print(is_non_empty(b'14) reviewsdata/pos_1.txt'))

tf.Tensor(True, shape=(), dtype=bool)


In [138]:
Text_ds = tf.data.Dataset.list_files("14) reviewsdata/*", shuffle=True).filter(is_non_empty)
print(type(Text_ds), count(Text_ds))
train_ds = Text_ds.take(int(count(Text_ds)*0.8))
test_ds = Text_ds.skip(int(count(Text_ds)*0.8))
print( "Train Data length: ",count(train_ds),"and Test Data length: ",count(test_ds),)
for i in Text_ds:
  print(i.numpy())

<class 'tensorflow.python.data.ops.filter_op._FilterDataset'> 4
Train Data length:  3 and Test Data length:  1
b'14) reviewsdata/pos_2.txt'
b'14) reviewsdata/pos_1.txt'
b'14) reviewsdata/neg_1.txt'
b'14) reviewsdata/neg_2.txt'


In [156]:
def get_label(file_path):
  parts = tf.strings.split(file_path, os.path.sep)
  if parts[-1].numpy().startswith(b"pos"):
    return "positive"
  else:
    return "negative"
def process_review(file_path):
  label = get_label(file_path)
  rev = tf.io.read_file(file_path)
  rev = tf.strings.lower(rev)
  rev = tf.strings.regex_replace(rev, "[^a-zA-Z]", " ")
  rev = tf.strings.regex_replace(rev, "\\s+", " ")
  rev = rev.numpy().decode("utf-8")
  return rev, label
j = 1
for i in Text_ds:
  print(f"Text and Label (Tuple) for File No.{j} => ",process_review(i))
  j += 1

Text and Label (Tuple) for File No.1 =>  ('this show was an amazing fresh innovative idea in the s when it first aired the first or years were brilliant but things dropped off after that by the show was not really funny anymore and it s continued its decline further to the complete waste of time it is today br br it s truly disgraceful how far this show has fallen the writing is painfully bad the performances are almost as bad if not for the mildly entertaining respite of the guest hosts this show probably wouldn t still be on the air i find it so hard to believe that the same creator that hand selected the original cast also chose the band of hacks that followed how can one recognize such brilliance and then see fit to replace it with such mediocrity i felt i must give stars out of respect for the original cast that made this show such a huge success as it is now the show is just awful i can t believe it s still on the air ', 'negative')
Text and Label (Tuple) for File No.2 =>  ('one 

In [158]:
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
def review_vectorized(DS):
  revs, labels = [], []
  for i in DS:
    rev, label = process_review(i)
    revs.append(rev)
    labels.append(label)
  tokenizer.fit_on_texts(revs)
  revs = tokenizer.texts_to_sequences(revs)
  revs = pad_sequences(revs, maxlen=100, padding="post")
  return revs, labels
revs, labels = review_vectorized(Text_ds)
for j, i in enumerate(revs, start=1):
    print(f"Padded Sequence No. {j}:", i)
print("Labels:", labels)
print("Word Index:", tokenizer.word_index)

Padded Sequence No. 1: [ 14   2 139 140 141   5   2 142 143  15  12 144  68  29  69  42  21   2
  70  10 145   8  34 146   9  71  13   2 147 148  13 149 150   2  72  73
 151 152   2 153   5 154  13 155  65  35  43 156  44 157   4  45  46 158
   9 159   8  11  44 160  10 161  10  74 162 163  47   5 164  14   2  72
  73  13 165  15  12  44   7 166 167  17   8   6 168   2  12   6  30 169
  10  35  29  71   8  16  69  21   2  70]
Padded Sequence No. 2: [  9  79 180   4  80   2  76   3   3 181  26   5  22  59  23 182 183   9
 184   7  81  23  74 185  41  32   7  82  27   7  52  17   7  52   2  51
   6 186  50  19 187 188 189  20 190  83   4  45 191  84  36  11  37  77
  53 192 193  22   2  81  10 194   9  46   7 195 196  51   4 197  10  85
   7  52  11 198 199  82 200   3   3  47   5  30  14   2  31 201  50 202
 203  17  14   2 204  11  36  30 205 206]
Padded Sequence No. 3: [ 25 288  10 289  29 100  10  25 290  14   8  28  17  10  85  98  10 291
   7 292  14  24   4 102 293   9   2  96 294